In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Generate synthetic X
num_subjects = 1000
num_features = 10

X_raw = torch.randn(num_subjects, num_features)


# 2. Add intercept column
ones_column = torch.ones(num_subjects, 1)

X = torch.cat((ones_column, X_raw), dim=1)
# X shape: (500, 6)


# 3. Define true beta
# 第一個是 intercept
beta_true = torch.tensor([
    1.0,   # intercept
    2.0,   # X1 effect
    -1.5,  # X2 effect
    0.5,   # X3 effect
    0.0,   # X4 no effect
    3.0,    # X5 effect
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
]).reshape(-1,1)


# 4. Generate Y according to linear model
noise = torch.randn(num_subjects,1) * 1

#Y_raw = X @ beta_true + noise  + X[:,4:5]*X[:,4:5] +X[:,2:3]*X[:,3:4]
Y_raw = X @ beta_true + noise  +X[:,2:3]*X[:,3:4]+ X[:,4:5]*X[:,4:5]

In [2]:
# 確認模擬資料是對的
beta = torch.linalg.solve(
    X.T @ X,
    X.T @ Y_raw
)

print(beta)

tensor([[ 2.0312e+00],
        [ 2.0127e+00],
        [-1.4149e+00],
        [ 5.3572e-01],
        [-1.1076e-01],
        [ 2.9987e+00],
        [-1.5756e-03],
        [-1.0583e-01],
        [-9.0591e-02],
        [ 4.0912e-02],
        [ 3.2417e-02]])


In [3]:
from sklearn.model_selection import train_test_split

# 建立 index
indices = torch.arange(num_subjects)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

# 切資料
X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]

Y_train = Y_raw[train_idx]
Y_test = Y_raw[test_idx]

In [4]:
# Train跟test都要加截距項
ones_train = torch.ones(X_train_raw.shape[0],1)
ones_test = torch.ones(X_test_raw.shape[0],1)


X_train = torch.cat(
    (ones_train, X_train_raw),
    dim=1
)

X_test = torch.cat(
    (ones_test, X_test_raw),
    dim=1
)

In [5]:
# 先把Y也加入X中，要讓attention matrix有Y的資訊
X_Y_train = torch.cat(
    (X_train, Y_train),
    dim=1
)

In [6]:
# 原始程式碼貼上(不訓練attention)

# Transpose data so the 6 features plus Y act as the "sequence" 
X_Y_features = X_Y_train.t()

# 5. Define the projection dimension (d_k)
d_k = 32
W_Q = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

W_K = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

# 6. Project features into Query (Q) and Key (K) spaces
Q = W_Q(X_Y_features)
K = W_K(X_Y_features)

# 7. Compute the raw attention scores
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

# 8. Scale by sqrt(d_k) and apply Softmax row-wise
attention_matrix = F.softmax(
    scores / (d_k ** 0.5),
    dim=-1
)

In [7]:
print("Attention Matrix Shape:", attention_matrix.shape)
print("\nAttention Matrix:\n", attention_matrix)

Attention Matrix Shape: torch.Size([12, 12])

Attention Matrix:
 tensor([[0.0799, 0.0478, 0.1115, 0.0719, 0.1013, 0.0749, 0.0714, 0.1167, 0.1008,
         0.0896, 0.1308, 0.0034],
        [0.0739, 0.0672, 0.0582, 0.1135, 0.0841, 0.0706, 0.0649, 0.1084, 0.0602,
         0.0757, 0.1364, 0.0869],
        [0.0447, 0.0976, 0.1238, 0.1046, 0.1260, 0.0555, 0.1054, 0.0547, 0.0854,
         0.1414, 0.0602, 0.0009],
        [0.0764, 0.1228, 0.0870, 0.0860, 0.0858, 0.0506, 0.1246, 0.0931, 0.0986,
         0.0778, 0.0694, 0.0279],
        [0.0446, 0.1235, 0.0590, 0.0467, 0.0622, 0.0734, 0.0856, 0.0624, 0.0808,
         0.0706, 0.0364, 0.2548],
        [0.0462, 0.1477, 0.0613, 0.0401, 0.0613, 0.0675, 0.0650, 0.0556, 0.0740,
         0.0736, 0.0295, 0.2780],
        [0.0615, 0.0979, 0.0850, 0.0755, 0.1079, 0.0685, 0.1037, 0.1575, 0.0548,
         0.0741, 0.0452, 0.0685],
        [0.0775, 0.0603, 0.1082, 0.0883, 0.1048, 0.0785, 0.1085, 0.1227, 0.0561,
         0.0697, 0.0937, 0.0316],
        [0.0556

In [8]:
# 把Y再從矩陣中拿掉
A = attention_matrix[:-1,:-1]

print("Attention Matrix Shape(拿掉Y):", A.shape)
print("\nAttention Matrix(拿掉Y):\n", A)

Attention Matrix Shape(拿掉Y): torch.Size([11, 11])

Attention Matrix(拿掉Y):
 tensor([[0.0799, 0.0478, 0.1115, 0.0719, 0.1013, 0.0749, 0.0714, 0.1167, 0.1008,
         0.0896, 0.1308],
        [0.0739, 0.0672, 0.0582, 0.1135, 0.0841, 0.0706, 0.0649, 0.1084, 0.0602,
         0.0757, 0.1364],
        [0.0447, 0.0976, 0.1238, 0.1046, 0.1260, 0.0555, 0.1054, 0.0547, 0.0854,
         0.1414, 0.0602],
        [0.0764, 0.1228, 0.0870, 0.0860, 0.0858, 0.0506, 0.1246, 0.0931, 0.0986,
         0.0778, 0.0694],
        [0.0446, 0.1235, 0.0590, 0.0467, 0.0622, 0.0734, 0.0856, 0.0624, 0.0808,
         0.0706, 0.0364],
        [0.0462, 0.1477, 0.0613, 0.0401, 0.0613, 0.0675, 0.0650, 0.0556, 0.0740,
         0.0736, 0.0295],
        [0.0615, 0.0979, 0.0850, 0.0755, 0.1079, 0.0685, 0.1037, 0.1575, 0.0548,
         0.0741, 0.0452],
        [0.0775, 0.0603, 0.1082, 0.0883, 0.1048, 0.0785, 0.1085, 0.1227, 0.0561,
         0.0697, 0.0937],
        [0.0556, 0.0824, 0.1093, 0.0731, 0.0752, 0.0544, 0.0783, 0.15

In [9]:
# 矩陣乘上Y的變異數
var_y = torch.var(Y_train)

A_var_y = A * var_y

In [10]:
A_var_y

tensor([[1.7240, 1.0315, 2.4068, 1.5505, 2.1863, 1.6154, 1.5399, 2.5188, 2.1761,
         1.9332, 2.8225],
        [1.5940, 1.4493, 1.2559, 2.4483, 1.8149, 1.5225, 1.4013, 2.3398, 1.2995,
         1.6337, 2.9426],
        [0.9644, 2.1065, 2.6716, 2.2562, 2.7179, 1.1968, 2.2739, 1.1806, 1.8421,
         3.0503, 1.2987],
        [1.6485, 2.6499, 1.8774, 1.8556, 1.8518, 1.0919, 2.6881, 2.0082, 2.1281,
         1.6786, 1.4969],
        [0.9618, 2.6642, 1.2736, 1.0082, 1.3431, 1.5839, 1.8462, 1.3455, 1.7441,
         1.5238, 0.7847],
        [0.9964, 3.1877, 1.3236, 0.8657, 1.3236, 1.4574, 1.4033, 1.1994, 1.5961,
         1.5886, 0.6364],
        [1.3280, 2.1122, 1.8331, 1.6299, 2.3280, 1.4772, 2.2374, 3.3984, 1.1817,
         1.5984, 0.9750],
        [1.6731, 1.3007, 2.3343, 1.9058, 2.2620, 1.6946, 2.3420, 2.6480, 1.2109,
         1.5046, 2.0208],
        [1.2005, 1.7789, 2.3579, 1.5769, 1.6225, 1.1746, 1.6903, 3.3132, 2.5603,
         2.0411, 2.1089],
        [2.3780, 2.1798, 2.2091, 1.71

In [11]:
X_train.T @ X_train

tensor([[800.0000,   5.6318,  -5.9632, -16.6760, -48.5696, -13.8970,  -5.5532,
          -4.7712,  23.0345, -13.1849,  53.3196],
        [  5.6318, 878.4042, -35.5583,  54.9601, -20.5034,  -9.8007,   8.9403,
           6.4887,  32.6690,  13.1710, -15.6376],
        [ -5.9632, -35.5583, 824.2017, -17.0627,  -1.4386, -57.0834, -11.2305,
          -7.4075, -45.5372,  26.4251, -42.2528],
        [-16.6760,  54.9601, -17.0627, 774.2184,  15.8301, -29.6834, -55.1373,
          39.1765, -20.0635,  61.8196,   4.5201],
        [-48.5696, -20.5034,  -1.4386,  15.8301, 879.3136,  23.6681,  22.8156,
          43.3949, -50.8642,  16.5220,  28.3631],
        [-13.8970,  -9.8007, -57.0834, -29.6834,  23.6681, 811.7356,  12.0902,
          13.9144,   6.9738, -17.2469, -10.6101],
        [ -5.5532,   8.9403, -11.2305, -55.1373,  22.8156,  12.0902, 831.9767,
          19.8740,  27.0497,   3.2835,  10.6903],
        [ -4.7712,   6.4887,  -7.4075,  39.1765,  43.3949,  13.9144,  19.8740,
         763.5792,

In [12]:
# Deep GLM 

p = X.shape[1]

I = torch.eye(
    p,
    dtype=X.dtype,
    device=X.device
)


beta_attention = torch.linalg.solve(X_train.T @ X_train + A + I*torch.trace(A), X_train.T @ Y_train)


In [13]:
#Ridge Regression

beta_ridge = torch.linalg.solve(X_train.T @ X_train + I, X_train.T @ Y_train)

Y_pred_ridge = X_test @ beta_ridge

# 算MSE
sse_ridge = torch.sum(
    (Y_test - Y_pred_ridge)**2
)

In [14]:
print(torch.linalg.det(A))
print(torch.linalg.det(X_train.T @ X_train))
print(torch.trace(A))
print(torch.trace(X_train.T @ X_train))
print(torch.linalg.det(A.T@A))

tensor(-2.7936e-14, grad_fn=<LinalgDetBackward0>)
tensor(1.1493e+32)
tensor(0.9715, grad_fn=<TraceBackward0>)
tensor(9106.9600)
tensor(7.8012e-28, grad_fn=<LinalgDetBackward0>)


In [15]:
# 把算出來的係數套入驗證集
Y_pred_attention = X_test @ beta_attention

# 算MSE
sse_attention = torch.sum(
    (Y_test - Y_pred_attention)**2
)

In [16]:
# OLS的標準解法
beta_ols = torch.linalg.solve(
    X_train.T @ X_train,
    X_train.T @ Y_train
)

print(
    "OLS beta:",
    beta_ols
)

Y_pred_ols = X_test @ beta_ols

sse_ols = torch.sum(
    (Y_test - Y_pred_ols)**2
)



OLS beta: tensor([[ 2.0634],
        [ 2.0264],
        [-1.4505],
        [ 0.5147],
        [-0.0538],
        [ 3.0723],
        [-0.0435],
        [-0.0745],
        [-0.1161],
        [ 0.0547],
        [ 0.0051]])


In [17]:
print(
    "Attention SSE:",
    sse_attention.item()
)

print("Ridge SSE:    ",
    sse_ridge.item()
     )

print(
    "OLS SSE:      ",
    sse_ols.item()
)

Attention SSE: 679.2371215820312
Ridge SSE:     679.3125
OLS SSE:       680.1807861328125
